# Exercício 6 · Bootstrap na prática: a fila da Cuida

**Disciplina 4 · Testes de Hipótese e Experimentação · Pós-graduação em Ciência de Dados e IA · UNIPDS**

## O contexto

A **Cuida** é um app de telemedicina. O paciente abre o app, descreve os sintomas numa triagem e entra numa fila até ser atendido por um médico em videochamada.

O time de Produto criou uma **triagem nova**, que faz mais perguntas no início e direciona o paciente para o médico certo. Para testar, os pacientes foram **sorteados**: metade passou pela triagem atual e metade pela nova.

O problema: tempo de espera é uma métrica muito assimétrica. A maioria espera poucos minutos, alguns esperam horas. E duas das perguntas do time não são sobre a média: são sobre a **mediana** e sobre o **percentil 90**, que não têm fórmula simples de intervalo de confiança. É trabalho para o **bootstrap**.

| Parte | Pergunta | Arquivo | Estatística |
|:---|:---|:---|:---|
| 1 | Quanto tempo o paciente típico espera hoje? A meta de 30 min é cumprida? | `espera_triagem.csv` | Mediana e percentil 90 de um grupo |
| 2 | A triagem nova reduz a espera do paciente típico? | `espera_triagem.csv` | Diferença das medianas |
| 3 | A triagem nova melhora o SLA (os 10% que mais esperam)? | `espera_triagem.csv` | Diferença dos percentis 90 |
| 4 | A triagem nova resolve mais consultas de primeira? | `consultas_resolucao.csv` | Taxa por consulta, com sorteio por paciente |

**Os dados**

`espera_triagem.csv`: uma linha por paciente sorteado.

| Coluna | Descrição |
|:---|:---|
| `paciente_id` | Identificador do paciente |
| `triagem` | `atual` ou `nova` |
| `espera_min` | Minutos entre o fim da triagem e o início da videochamada |

`consultas_resolucao.csv`: uma linha por **consulta**, durante um mês. Um mesmo paciente pode ter várias consultas.

| Coluna | Descrição |
|:---|:---|
| `consulta_id` | Identificador da consulta |
| `paciente_id` | Paciente que fez a consulta (a unidade sorteada) |
| `triagem` | `atual` ou `nova` |
| `resolvida` | 1 se o problema foi resolvido nessa consulta, sem precisar de retorno; 0 caso contrário |

### Como usar este notebook

Cada seção traz um título com o que precisa ser feito. Escreva o código na célula vazia abaixo de cada título (se precisar de mais células, crie) e, depois de rodar, anote em texto a sua leitura do resultado: o que o número diz sobre a pergunta de negócio.

**Todos os intervalos são de 95%. Nos bootstraps, use 5.000 reamostras e semente 42 (`np.random.default_rng(42)`)**, para comparar seus números com os do gabarito.

## Parte 0 · Preparação

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

N_REAMOSTRAS = 5000

## Parte 1 · O retrato da espera hoje

Antes de comparar as triagens, olhe só o grupo **atual**. A Cuida promete no app que **90% dos pacientes são atendidos em até 30 minutos**. Em linguagem estatística: o percentil 90 da espera deveria ser de no máximo 30 min.

**A receita do bootstrap.** (1) Sorteie, com reposição, uma amostra do mesmo tamanho da original; (2) calcule nela a estatística que interessa; (3) repita milhares de vezes; (4) os percentis 2,5 e 97,5 dessas estatísticas formam o IC de 95%. Serve para qualquer estatística, inclusive as que não têm fórmula.

### 1.1 Os dados e a distribuição da espera

### 1.2 IC da mediana por bootstrap (e a média, para comparar)

### 1.3 IC do percentil 90 e a meta de 30 minutos

## Parte 2 · A triagem nova reduz a espera do paciente típico?

- **Parâmetro:** diferença das medianas, nova − atual.
- **Bootstrap para dois grupos:** reamostre **cada grupo separadamente**, com o tamanho dele, e calcule a diferença em cada rodada. O sorteio do experimento separou os grupos; a reamostragem respeita essa separação.

### 2.1 Bootstrap da diferença das medianas

### 2.2 E a média? Teste t de Welch

## Parte 3 · O SLA: e os 10% que mais esperam?

A meta do app fala do percentil 90, não da mediana. Uma triagem pode melhorar o paciente típico e piorar a cauda, ou o contrário.

- **Parâmetro:** diferença dos percentis 90, nova − atual.

### 3.1 Bootstrap da diferença dos percentis 90

### 3.2 O que acontece na cauda da triagem nova?

## Parte 4 · Resolução na primeira consulta: quem foi sorteado?

A métrica é a **taxa de resolução**: consultas resolvidas sem retorno ÷ total de consultas. Mas o sorteio foi feito por **paciente**, e um mesmo paciente faz várias consultas. Consultas do mesmo paciente se parecem: quem tem um problema simples resolve quase todas; quem tem um caso difícil volta várias vezes.

Isso muda o bootstrap: a reamostragem precisa imitar o sorteio. Se a unidade sorteada é o paciente, reamostramos **pacientes**, levando junto **todas as consultas** de cada um. É o **bootstrap por cluster**.

### 4.1 Os dados por consulta e por paciente

### 4.2 Bootstrap reamostrando consultas

### 4.3 Bootstrap por cluster: reamostrando pacientes

## Parte 5 · As recomendações

*Escreva aqui as suas recomendações para cada pergunta do time.*

### Perguntas para discussão

1. Por que, na Parte 2, a mediana e a média chegam a conclusões diferentes? Qual das duas responde à pergunta do time?
2. Por que o IC do percentil 90 é tão mais largo que o IC da mediana, com os mesmos 220 pacientes?
3. Na Parte 4, por que reamostrar consultas dá um IC mais estreito? Que erro de decisão isso poderia causar?
4. O bootstrap resolve qualquer problema de amostra? Cite uma situação em que ele não ajuda.